# 🔥 组织经验淬炼师 | KnowledgeForge
**GDG Shanghai Gemma 4 Hackathon 2026 | Track A: AI Agent**

## FIRE 经验萃取分析法
| 层级 | 英文 | 中文 | 核心目标 |
|------|------|------|----------|
| F | Function | 职能领域 | 识别专家核心职责范围 |
| I | Instance | 事件实例 | 还原真实行为事件（STAR）|
| **R** | **Reasoning** | **推理判断** | **挖掘决策逻辑（核心创新层）** |
| E | Essence | 底层原则 | 提炼可迁移的心智模式 |

In [ ]:
!pip install google-genai -q

In [ ]:
import os, json
os.environ['GEMINI_API_KEY'] = 'YOUR_API_KEY_HERE'

In [ ]:
from google import genai
from google.genai import types

MODEL = 'models/gemma-4-26b-a4b-it'
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
skill_library = {}

# FIRE System Prompt
SYSTEM_PROMPT = """<|think|>
你是首席经验淬炼引导师，由 KnowledgeForge 构建。
你掌握 FIRE 经验萃取分析法：

F层·职能识别：「你最核心的职责是什么？」
I层·事件还原：「能讲一个最具挑战的具体案例吗？」
R层·判断挖掘★：「你当时怎么判断要这么做？脑子里第一个念头是什么？」
E层·原则提炼：「用一句话总结你的判断标准是什么？」

铁律：每次只问一个问题。专家说「我们一般怎么做」时立刻拉回具体事件。
当 FIRE 四层信息充分时，主动提出保存技能卡片。"""

TOOLS = [types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name='save_skill_card',
        description='仅当访谈完成且用户确认时调用，保存 FIRE 结构化技能卡片',
        parameters={
            'type': 'object',
            'properties': {
                'skill_name':          {'type': 'string'},
                'fire_function':       {'type': 'string'},
                'fire_instance':       {'type': 'string'},
                'fire_reasoning':      {'type': 'array', 'items': {'type': 'string'}},
                'fire_essence':        {'type': 'string'},
                'core_steps':          {'type': 'array', 'items': {
                    'type': 'object',
                    'properties': {
                        'step': {'type': 'integer'},
                        'action': {'type': 'string'},
                        'key_decision': {'type': 'string'}
                    }
                }},
                'applicable_scenarios':{'type': 'array', 'items': {'type': 'string'}}
            },
            'required': ['skill_name','fire_function','fire_instance',
                         'fire_reasoning','fire_essence','core_steps','applicable_scenarios']
        }
    )
])]

def extract_answer(response):
    """铁律：过滤思考块，只返回最终答案"""
    parts = response.candidates[0].content.parts
    return ''.join(p.text for p in parts if not p.thought and p.text).strip()

def chat(history, user_message):
    history.append({'role': 'user', 'parts': [{'text': user_message}]})
    response = client.models.generate_content(
        model=MODEL, contents=history,
        config=types.GenerateContentConfig(
            temperature=1.0, top_p=0.95, top_k=64, max_output_tokens=4096,
            tools=TOOLS, system_instruction=SYSTEM_PROMPT
        )
    )
    parts = response.candidates[0].content.parts
    tool_calls = [p for p in parts if hasattr(p, 'function_call') and p.function_call]
    if tool_calls:
        for p in tool_calls:
            fc = p.function_call
            skill_id = f'SKILL-{len(skill_library)+1:03d}'
            skill_library[skill_id] = dict(fc.args)
            print(f'\n[工具调用] {fc.name} → {skill_id}')
            print('[FIRE 技能卡片]')
            print(json.dumps(dict(fc.args), indent=2, ensure_ascii=False))
    answer = extract_answer(response)
    history.append({'role': 'model', 'parts': [{'text': answer}]})
    return answer

print('✅ KnowledgeForge Agent 就绪')
print(f'模型：{MODEL}')
print('方法论：FIRE 经验萃取分析法')

## Demo：FIRE 四层访谈完整演示
模拟萃取销售总监的大客户谈判经验

In [ ]:
# F层：职能识别
history = []
r = chat(history, '我想提炼我们销售总监老王的大客户谈判经验。')
print('引导师：', r)

In [ ]:
# I层：事件实例还原
r = chat(history, '一个大型国企客户，合同800万，采购部门一直压价，最终我们以原价签约。')
print('引导师：', r)

In [ ]:
# R层：推理判断挖掘（核心）
r = chat(history, '转折点是老王停止为价格辩护，转而问客户：如果这个采购失败，你的项目会怎样？这句话重新定义了谈判。')
print('引导师：', r)

In [ ]:
# E层：底层原则 + 保存
r = chat(history, '他说他的判断标准是：当对方开始反复强调价格，说明他们已经认可了价值，这时候要把话题引到「不买的代价」而不是「买的成本」。请保存为技能卡片。')
print('引导师：', r)

In [ ]:
# 展示 FIRE 知识库
print(f'\n📚 FIRE 知识库（已保存 {len(skill_library)} 个技能）：')
for sid, skill in skill_library.items():
    print(f'\n{sid}: {skill["skill_name"]}')
    print(f'  F · 职能：{skill.get("fire_function","")}')
    print(f'  I · 场景：{skill.get("fire_instance","")}')
    print(f'  R · 判断：{skill.get("fire_reasoning","")}')
    print(f'  E · 原则：{skill.get("fire_essence","")}')